# Scaling Laws

Given a fixed compute budget, how large a model should you train, and on how much data?
Before 2020 that was taste. Scaling laws made it arithmetic: model loss follows a
predictable power law in parameters, data and compute, and the relationship is regular
enough to **extrapolate from small runs to large ones**.

That predictability is what makes large-scale training an engineering discipline rather
than a gamble — you can spend 1% of your budget on a sweep of small models, fit a curve,
and know roughly what the big run will produce before committing to it.

Second topic in the [Pre-training](tokenization-bpe.ipynb) track.

## 1. What & Why

The central empirical finding: test loss falls as a **power law** in each of the three
scaling axes, over many orders of magnitude.

```
L(N) ≈ L∞ + (Nc / N)^αN        N = parameters
L(D) ≈ L∞ + (Dc / D)^αD        D = training tokens
```

`L∞` is the irreducible loss — the entropy of the data itself, which no model can beat.
A power law is a **straight line on a log-log plot**, which is why every scaling paper
plots that way and why the fit is so easy to check by eye.

The practically important question is the **compute-optimal allocation**. Compute is
roughly `C ≈ 6ND`, so for a fixed `C` you can trade `N` against `D`. The two landmark
answers disagreed:

- **Kaplan et al. (2020)** concluded you should scale parameters much faster than data —
  which is why models of that era were large and comparatively under-trained.
- **Chinchilla (Hoffmann et al., 2022)** re-did the analysis with the learning-rate
  schedule handled correctly and found `N` and `D` should scale **roughly equally**: about
  **20 tokens per parameter**. A 70B model trained on 1.4T tokens beat a 280B model
  trained on 300B tokens, using the same compute.

**And the twist that governs practice today:** compute-optimal is optimal for *training*.
If you will serve the model many times, it is rational to train a **smaller model on more
data than Chinchilla prescribes** — you pay more to train, and less forever after.

## 2. Mental Model

**A straight line on log-log paper, plus a budget constraint.**

Two pictures, and you need both:

The first is the **power law itself**: plot loss against compute on log-log axes and you
get a line. A line is extrapolable, and that is the entire practical value — you fit on
models you can afford and read off the one you cannot.

The second is the **budget constraint**. `C ≈ 6ND` means that on a log-log plot of `N`
against `D`, a fixed compute budget is a straight line with slope −1. Every point on that
line is a model you could train for the same money: tall-and-thin (big model, little
data) at one end, short-and-wide (small model, lots of data) at the other. Loss is a
surface over that plane, and the compute-optimal point is where the budget line touches
the lowest contour.

Kaplan and Chinchilla did not disagree about the existence of that tangent point — they
disagreed about **where it is**, and the reason was a methodological detail (whether the
learning-rate schedule matched the token count of each run). That is worth remembering as
a caution: these are *empirical fits*, and they are only as good as the experimental
design behind them.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Power law** | `L ≈ L∞ + (x_c/x)^α`. Straight on log-log. The empirical form loss takes in `N`, `D` and `C`. |
| **Irreducible loss `L∞`** | The data's own entropy. Fitting without this term makes the law look wrong at scale. |
| **`C ≈ 6ND`** | Training FLOPs ≈ 6 × parameters × tokens. Two for the forward pass, four for the backward. |
| **Compute-optimal** | The `(N, D)` split minimising loss at fixed `C`. |
| **Chinchilla ratio** | ≈ **20 tokens per parameter** at the compute-optimal point. |
| **Over-training** | Deliberately exceeding that ratio to get a smaller model of the same quality. Standard for models that will be served. |
| **Inference-aware scaling** | Choosing `(N, D)` to minimise *lifetime* cost — training plus all inference — rather than training loss. |
| **Emergence** | Capabilities that appear abruptly with scale. Often an artefact of a discontinuous metric; smooth on continuous ones. |
| **Data-constrained scaling** | What to do when you run out of unique tokens: repeat (up to ~4 epochs is nearly as good as fresh data), then diminishing returns. |
| **Extrapolation risk** | Fits are valid inside their measured range. Predictions far outside it are conjecture. |

## 4. Setup

NumPy. Every example fits a law to synthetic runs generated from a *known* law, so the
recovered parameters can be checked against the truth — the only honest way to show that
a fitting procedure works.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(suppress=True)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — fit a power law, and see why the irreducible term matters

Generate runs from a known law, add realistic noise, and recover the parameters.

In [2]:
TRUE = dict(L_inf=1.70, N_c=8.8e13, alpha=0.076)     # Kaplan-like values

def loss_of_N(N, L_inf, N_c, alpha):
    return L_inf + (N_c / N) ** alpha

sizes = np.array([1e6, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9])
observed = loss_of_N(sizes, **TRUE) * (1 + rng.normal(0, 0.004, len(sizes)))

def fit_with_floor(N, L, grid=None):
    '''Fit L = L_inf + (N_c/N)^alpha by scanning L_inf and least-squares in log space.'''
    best = None
    for L_inf in (grid if grid is not None else np.linspace(0.0, min(L) * 0.999, 400)):
        y = np.log(L - L_inf)
        A = np.vstack([np.ones_like(N), -np.log(N)]).T
        coef, *_ = np.linalg.lstsq(A, y, rcond=None)
        resid = float(np.sum((A @ coef - y) ** 2))
        if best is None or resid < best[0]:
            best = (resid, L_inf, float(coef[1]), float(np.exp(coef[0] / coef[1])))
    _, L_inf, alpha, N_c = best
    return L_inf, N_c, alpha

L_inf, N_c, alpha = fit_with_floor(sizes, observed)
print("fitted WITH an irreducible-loss term:")
print(f"  L_inf = {L_inf:.3f}  (true {TRUE['L_inf']:.3f})")
print(f"  alpha = {alpha:.4f}  (true {TRUE['alpha']:.4f})")
print("  -- close but not exact, and that is the honest result: L_inf and alpha trade")
print("     off against each other, so the floor is only well identified once your")
print("     largest runs are near it. These runs are all far above it.")

# Now the common mistake: assume the loss goes to zero.
y = np.log(observed)
A = np.vstack([np.ones_like(sizes), -np.log(sizes)]).T
coef, *_ = np.linalg.lstsq(A, y, rcond=None)
alpha_naive = float(coef[1])
print(f"\nfitted WITHOUT the L_inf term:")
print(f"  alpha = {alpha_naive:.4f}   <- badly wrong")

print(f"\n{'N':>12} {'observed':>10} {'with L_inf':>12} {'without':>10}")
for n, o in zip(sizes, observed):
    with_ = loss_of_N(n, L_inf, N_c, alpha)
    without = np.exp(coef[0]) * n ** (-alpha_naive)
    print(f"{n:12.0e} {o:10.4f} {with_:12.4f} {without:10.4f}")

print(f"\n{'extrapolate to N':>18} {'with L_inf':>12} {'without':>10} {'true':>8}")
for big in (1e12, 1e15, 1e18, 1e21):
    with_ = loss_of_N(big, L_inf, N_c, alpha)
    without = np.exp(coef[0]) * big ** (-alpha_naive)
    print(f"{big:18.0e} {with_:12.4f} {without:10.4f} {loss_of_N(big, **TRUE):8.4f}")

print(f"\nThe floorless fit keeps descending forever and crosses the true irreducible")
print(f"loss of {TRUE['L_inf']:.2f} somewhere past 1e18 -- predicting a model that")
print("compresses the data better than its own entropy allows, which is impossible.")
print("\nAlways fit the floor. A pure power law says loss reaches zero with enough")
print("parameters; it cannot, because the data itself is not deterministic.")

fitted WITH an irreducible-loss term:
  L_inf = 2.010  (true 1.700)
  alpha = 0.0837  (true 0.0760)
  -- close but not exact, and that is the honest result: L_inf and alpha trade
     off against each other, so the floor is only well identified once your
     largest runs are near it. These runs are all far above it.

fitted WITHOUT the L_inf term:
  alpha = 0.0486   <- badly wrong

           N   observed   with L_inf    without
       1e+06     5.7188       5.7208     5.6894
       3e+06     5.3913       5.3948     5.3938
       1e+07     5.0842       5.0702     5.0874
       3e+07     4.8031       4.8013     4.8230
       1e+08     4.5203       4.5337     4.5491
       3e+08     4.3095       4.3119     4.3127
       1e+09     4.0969       4.0912     4.0677

  extrapolate to N   with L_inf    without     true
             1e+12       3.1773     2.9082   3.1053
             1e+15       2.6648     2.0792   2.5313
             1e+18       2.3774     1.4866   2.1918
             1e+21   

### Example 2 — the compute-optimal frontier, and the Chinchilla ratio

For a fixed budget `C ≈ 6ND`, sweep the split between parameters and data and find the
minimum. This is the calculation Chinchilla made, in miniature.

In [3]:
# A Chinchilla-style joint law: loss depends on both N and D, each with its own exponent.
E, A_, B_, a_, b_ = 1.69, 406.4, 410.7, 0.34, 0.28

def joint_loss(N, D):
    return E + A_ / N**a_ + B_ / D**b_

def optimal_split(C, n_grid=400):
    '''Sweep N along the budget line D = C/(6N) and return the best split.'''
    Ns = np.logspace(7, 12, n_grid)
    Ds = C / (6 * Ns)
    losses = joint_loss(Ns, Ds)
    i = int(np.argmin(losses))
    return Ns[i], Ds[i], losses[i]

print(f"{'compute (FLOPs)':>16} {'optimal N':>12} {'optimal D':>13} {'tokens/param':>13} "
      f"{'loss':>7}")
for C in (1e19, 1e20, 1e21, 1e22, 1e23, 1e24):
    N, Dt, L = optimal_split(C)
    print(f"{C:16.0e} {N:12.2e} {Dt:13.2e} {Dt/N:13.1f} {L:7.3f}")

print("\nTwo things to read here, and the second is the more useful one.")
print("\nFirst: N and D grow TOGETHER, each roughly as a power of compute. That is the")
print("core Chinchilla result and it is what overturned the earlier practice of")
print("scaling parameters far faster than data.")
print("\nSecond: the tokens-per-parameter ratio above is NOT the famous 20:1, and it")
print("drifts upward with scale. That is not a bug in this notebook -- these are the")
print("published constants of Chinchilla's own parametric fit (their Approach 3), and")
print("they genuinely do not reproduce the 20:1 headline that came from their")
print("Approaches 1 and 2. The replication attempt cited in Resources found errors in")
print("that fit and reported wide confidence intervals on these exponents.")
print("\nSo: trust the SHAPE (scale N and D together), treat the exact constant as")
print("uncertain, and re-fit on your own data and architecture before betting on it.")

print("\nWhat it costs to get the split wrong, at a fixed budget of 1e22 FLOPs:")
C = 1e22
N_opt, D_opt, L_opt = optimal_split(C)
print(f"{'strategy':28} {'N':>10} {'D':>11} {'tok/param':>10} {'loss':>7} {'excess':>8}")
for label, ratio in [("10x too large (undertrained)", 10.0), ("2x too large", 2.0),
                     ("compute-optimal", 1.0), ("2x too small", 0.5),
                     ("10x too small (overtrained)", 0.1)]:
    N = N_opt * ratio
    Dt = C / (6 * N)
    L = joint_loss(N, Dt)
    print(f"{label:28} {N:10.2e} {Dt:11.2e} {Dt/N:10.1f} {L:7.3f} {L - L_opt:+8.3f}")

print("\nNote the asymmetry: being too LARGE costs more than being too small by the same")
print("factor. Under-training a big model wastes compute the model cannot use, which is")
print("precisely the error the pre-Chinchilla generation of models made.")

 compute (FLOPs)    optimal N     optimal D  tokens/param    loss
           1e+19     2.26e+08      7.39e+09          32.7   2.986
           1e+20     6.38e+08      2.61e+10          41.0   2.600
           1e+21     1.80e+09      9.25e+10          51.4   2.329
           1e+22     5.09e+09      3.27e+11          64.3   2.139
           1e+23     1.48e+10      1.13e+12          76.0   2.005
           1e+24     4.18e+10      3.98e+12          95.2   1.911

Two things to read here, and the second is the more useful one.

First: N and D grow TOGETHER, each roughly as a power of compute. That is the
core Chinchilla result and it is what overturned the earlier practice of
scaling parameters far faster than data.

Second: the tokens-per-parameter ratio above is NOT the famous 20:1, and it
drifts upward with scale. That is not a bug in this notebook -- these are the
published constants of Chinchilla's own parametric fit (their Approach 3), and
they genuinely do not reproduce the 20:1 headl

### Example 3 — compute-optimal is the wrong objective if you will serve the model

Chinchilla minimises training loss per training FLOP. If the model then serves billions
of tokens, the right objective is **total lifetime cost**, and that pushes you toward
smaller, deliberately over-trained models.

In [4]:
def lifetime_flops(N, D, inference_tokens):
    '''Training ~6ND, inference ~2N per token generated.'''
    return 6 * N * D, 2 * N * inference_tokens

TARGET_LOSS = 2.05
C_BUDGET = 1e22

# Find every (N, D) pair reaching the target loss, then cost each over a serving lifetime.
Ns = np.logspace(9, 11.5, 300)
rows = []
for N in Ns:
    # solve joint_loss(N, D) = TARGET for D
    rem = TARGET_LOSS - E - A_ / N**a_
    if rem <= 0:
        continue                       # this N cannot reach the target at ANY data size
    Dt = (B_ / rem) ** (1 / b_)
    if Dt > 1e14:
        continue                       # beyond any plausible token budget -- drop it
    rows.append((N, Dt))

print(f"models that all reach loss {TARGET_LOSS}:\n")
print(f"{'N':>11} {'D':>11} {'tok/param':>10} {'train FLOPs':>13} "
      f"{'+1e12 served':>14} {'+1e14 served':>14}")
for N, Dt in rows[::60]:
    tr, _ = lifetime_flops(N, Dt, 0)
    _, inf12 = lifetime_flops(N, Dt, 1e12)
    _, inf14 = lifetime_flops(N, Dt, 1e14)
    print(f"{N:11.2e} {Dt:11.2e} {Dt/N:10.0f} {tr:13.2e} {tr+inf12:14.2e} "
          f"{tr+inf14:14.2e}")

best_train = min(rows, key=lambda r: 6 * r[0] * r[1])
best_1e14 = min(rows, key=lambda r: 6 * r[0] * r[1] + 2 * r[0] * 1e14)
print(f"\ncheapest to TRAIN            : N = {best_train[0]:.2e}, "
      f"{best_train[1]/best_train[0]:.0f} tokens/param")
print(f"cheapest over 1e14 served tokens: N = {best_1e14[0]:.2e}, "
      f"{best_1e14[1]/best_1e14[0]:.0f} tokens/param")

print("\nOnce serving costs dominate, the optimum moves to a SMALLER model trained on")
print("far more tokens than any training-optimal rule prescribes -- the exact figure")
print("scales with how much you expect to serve, and 1e14 tokens is a heavy assumption.")
print("That is the reasoning behind current open models being trained at hundreds or")
print("thousands of tokens per parameter: they are optimised for the total bill, not")
print("for the training run.")

models that all reach loss 2.05:

          N           D  tok/param   train FLOPs   +1e12 served   +1e14 served
   1.47e+09    9.93e+13      67565      8.76e+23       8.79e+23       1.17e+24
   4.67e+09    1.88e+12        402      5.25e+22       6.18e+22       9.86e+23
   1.48e+10    4.94e+11         33      4.39e+22       7.35e+22       3.01e+24
   4.70e+10    2.50e+11          5      7.04e+22       1.64e+23       9.47e+24
   1.49e+11    1.68e+11          1      1.50e+23       4.49e+23       3.00e+25

cheapest to TRAIN            : N = 9.89e+09, 71 tokens/param
cheapest over 1e14 served tokens: N = 2.20e+09, 5460 tokens/param

Once serving costs dominate, the optimum moves to a SMALLER model trained on
far more tokens than any training-optimal rule prescribes -- the exact figure
scales with how much you expect to serve, and 1e14 tokens is a heavy assumption.
That is the reasoning behind current open models being trained at hundreds or
thousands of tokens per parameter: they are optim

### Example 4 — how far can you trust an extrapolation?

The honest limitation. Fit on a narrow range of small models, predict a large one, and
measure the error.

In [5]:
def experiment(fit_max, noise=0.004, seed=0):
    r = np.random.default_rng(seed)
    train_sizes = np.array([s for s in [1e6, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9, 3e9]
                            if s <= fit_max])
    obs = loss_of_N(train_sizes, **TRUE) * (1 + r.normal(0, noise, len(train_sizes)))
    L_inf, N_c, alpha = fit_with_floor(train_sizes, obs)
    target = 1e12
    pred = loss_of_N(target, L_inf, N_c, alpha)
    truth = loss_of_N(target, **TRUE)
    return len(train_sizes), pred, truth

print(f"predicting the loss of a 1e12-parameter model:\n")
print(f"{'fit up to':>12} {'runs':>5} {'predicted':>11} {'true':>8} {'error':>8} "
      f"{'orders extrapolated':>21}")
for fit_max in (1e7, 1e8, 1e9, 3e9):
    n_runs, pred, truth = experiment(fit_max)
    print(f"{fit_max:12.0e} {n_runs:5d} {pred:11.4f} {truth:8.4f} "
          f"{pred-truth:+8.4f} {np.log10(1e12/fit_max):21.1f}")

print("\nExtrapolating five orders of magnitude from three runs is unreliable; the same")
print("prediction from a wider fitting range is much better. The error shrinks with")
print("both the number of runs and the SPAN they cover -- span matters more.")

print("\nSensitivity to measurement noise (fitting up to 1e9):")
for noise in (0.002, 0.01, 0.03):
    preds = [experiment(1e9, noise, seed=s)[1] for s in range(12)]
    print(f"  noise {noise:.1%}: predictions span {min(preds):.3f} - {max(preds):.3f}")

print("\nSmall errors in the exponent compound enormously over orders of magnitude, so")
print("scaling-law predictions need tight loss measurements and a wide fitting range.")
print("Treat a prediction more than ~2 orders beyond your data as a hypothesis, and")
print("check it with an intermediate run before betting a cluster on it.")

predicting the loss of a 1e12-parameter model:

   fit up to  runs   predicted     true    error   orders extrapolated
       1e+07     3      3.7136   3.1053  +0.6083                   5.0
       1e+08     5      2.8428   3.1053  -0.2625                   4.0
       1e+09     7      3.1773   3.1053  +0.0720                   3.0
       3e+09     8      3.1507   3.1053  +0.0454                   2.5

Extrapolating five orders of magnitude from three runs is unreliable; the same
prediction from a wider fitting range is much better. The error shrinks with
both the number of runs and the SPAN they cover -- span matters more.

Sensitivity to measurement noise (fitting up to 1e9):
  noise 0.2%: predictions span 3.023 - 3.146
  noise 1.0%: predictions span 2.803 - 3.161
  noise 3.0%: predictions span 2.634 - 3.106

Small errors in the exponent compound enormously over orders of magnitude, so
scaling-law predictions need tight loss measurements and a wide fitting range.
Treat a prediction mor

## 6. Gotchas & Pitfalls

- **Fitting without an irreducible-loss term.** Example 1. A pure power law predicts zero
  loss at infinite scale, which is impossible, and the misfit distorts the exponent.
- **Extrapolating far beyond the fitted range.** Example 4. Two orders of magnitude is
  ambitious; five is a guess.
- **Mismatching the learning-rate schedule to the token count.** This is precisely what
  made Kaplan's and Chinchilla's conclusions differ. A cosine schedule tuned for a
  different horizon makes short runs look artificially bad.
- **Using `C = 6ND` outside its assumptions.** It ignores attention's quadratic term
  (material at long context), activation recomputation, and MoE sparsity. For MoE, `N` is
  *active* parameters, not total.
- **Optimising training loss when you will serve the model.** Example 3.
- **Assuming the law transfers across data distributions.** The constants are properties
  of your corpus. A better dataset shifts the whole curve; the exponents are more stable
  than the constants but neither is universal.
- **Ignoring the data wall.** These laws assume fresh tokens. Repeating data works
  surprisingly well for a few epochs and then decays sharply.
- **Reading too much into emergence.** Sharp capability jumps are frequently artefacts of
  a discontinuous metric (exact-match accuracy); the underlying log-likelihood usually
  improves smoothly.
- **Comparing loss across tokenizers.** Loss is per token, so a different tokenizer
  changes the number without changing the model's quality. Compare bits-per-byte.

## 7. When to Use vs Alternatives

| Question | Approach |
|---|---|
| How big a model for my budget? | **Chinchilla-style sweep** (Example 2) on your own data |
| I will serve this a lot | **Inference-aware scaling** (Example 3) — over-train a smaller model |
| Will this architecture change help at scale? | Fit laws for **both** architectures on small models and compare exponents, not single points |
| I have limited unique data | **Data-constrained scaling laws** — repeat up to a few epochs, then expect decay |
| Will capability X appear at scale? | Scaling laws predict **loss**, not capabilities. Use a continuous proxy metric |
| Mixture-of-Experts | MoE-specific laws — `N` is active parameters, and the sparsity ratio is its own axis |

**The honest position.** Scaling laws are an empirical regularity, not a theory. They fit
remarkably well within the range they were measured over, the constants are
dataset-specific, and both landmark papers were revised by later work — Chinchilla's own
numbers have been re-analysed and adjusted.

Use them the way they earn their keep: to make a **relative** decision (this split beats
that split, this architecture scales better) from cheap small runs, rather than to predict
an absolute number many orders of magnitude away. And when the prediction matters, buy an
intermediate-scale run to check the curve before committing the full budget.

## 8. Resources

- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) — Kaplan et al., 2020. The original; still the clearest exposition of the methodology.
- [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556) — Hoffmann et al., 2022 (Chinchilla). The 20-tokens-per-parameter result and what Kaplan got wrong.
- [Chinchilla Scaling: A replication attempt](https://arxiv.org/abs/2404.10102) — a careful re-analysis finding errors in the original fits; a good lesson in treating these as empirical.
- [Beyond Chinchilla-Optimal: Accounting for Inference in Language Model Scaling Laws](https://arxiv.org/abs/2401.00448) — Example 3, developed properly.
- [Scaling Data-Constrained Language Models](https://arxiv.org/abs/2305.16264) — what happens when you run out of unique tokens; repeated-data scaling laws.
- [Are Emergent Abilities of Large Language Models a Mirage?](https://arxiv.org/abs/2304.15004) — the metric-discontinuity argument against naive readings of emergence.
- [Scaling Laws for Autoregressive Generative Modeling](https://arxiv.org/abs/2010.14701) — the same power-law form across images, video, and maths, which is the evidence that it is not a language-specific quirk.

What practical capability do scaling laws provide that makes large-scale training an engineering discipline rather than a gamble?

Explain the compute-budget constraint geometrically, and what the compute-optimal point is in that picture. Use the relation C ≈ 6ND.

Why must a scaling-law fit include an irreducible-loss term L∞?

In [ ]:
def compute_flops(n_params, n_tokens):
    ...

def tokens_for_budget(n_params, flops):
    ...


In [ ]:
assert compute_flops(70_000_000_000, 1_400_000_000_000) == 6 * 70 * 10**9 * 14 * 10**11
assert compute_flops(1, 1) == 6
N = 7 * 10**9
C = 10**22
D = tokens_for_budget(N, C)
assert abs(compute_flops(N, D) - C) / C < 1e-9, 'must round-trip'
assert tokens_for_budget(2 * N, C) < tokens_for_budget(N, C), 'bigger model, fewer tokens'
assert abs(tokens_for_budget(N, 2 * C) / tokens_for_budget(N, C) - 2.0) < 1e-9


You are told a model should be trained at 20 tokens per parameter. When is that the wrong target, and why?

How much should you trust a scaling-law extrapolation, and what is the appropriate way to use these laws in practice?